# TTS Voice Cloning Comparison: VoxCPM2 vs Chatterbox Multilingual

Explore and A/B test both models using Carlo's voice sample(s).

**Goal:** ElevenLabs-like voice cloning quality, fast on Apple Silicon (M4, MPS backend), across EN / FR / IT / ES.

**Workflow:**
1. Set up environment
2. Load Carlo's reference audio (Italian clip is the primary one)
3. Generate with VoxCPM2
4. Generate with Chatterbox Multilingual
5. Benchmark speed (real-time factor)
6. Listen side by side and decide

> Tip: run cross-lingual generation first (Italian reference -> EN/FR/ES output) before asking for more recordings in other languages.

## 1. Setup

Run once. Uses MPS (Metal) as the device for both models on Apple Silicon.

In [1]:
# Core packages for both models
%pip install -q voxcpm chatterbox-tts torch torchaudio soundfile ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 18.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 111.2 MB/s eta 0:00:00


In [2]:
import time
import os
from pathlib import Path

import torch
import soundfile as sf
from IPython.display import Audio, display, Markdown

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Using device: cuda


## 2. Config: reference audio + test sentences

Fill in the paths to Carlo's recordings below. At minimum you need `it` (Italian) — the others are optional if you're testing cross-lingual generation from the Italian reference alone.

In [ ]:
# --- Fill these in with the actual paths to Carlo's recordings ---
REFERENCE_AUDIO = {
    "it": "./samples/carlo_it.wav",   # required
    "en": "./samples/carlo_en.wav",   # optional
    "fr": "./samples/carlo_fr.wav",   # optional
    "es": "./samples/carlo_es.wav",   # optional
}

# Exact transcript of the Italian reference clip — improves VoxCPM2's "Ultimate Cloning" mode
REFERENCE_TRANSCRIPT_IT = (
    "Buongiorno, spero che tu stia bene oggi. Volevo darti un rapido aggiornamento "
    "sullo stato del progetto. Tutto sta procedendo molto bene, e penso che saremo "
    "pronti a mostrarti i risultati molto presto. Ti sembra una buona notizia?"
)

# Test sentences to synthesize in the cloned voice (same content as the reference,
# so you can judge cloning fidelity fairly against the original)
TEST_SENTENCES = {
    "it": "Buongiorno, spero che tu stia bene oggi. Volevo darti un rapido aggiornamento sullo stato del progetto.",
    "en": "Good morning, I hope you're doing well today. I wanted to give you a quick update on where things stand.",
    "fr": "Bonjour, j'espère que vous allez bien aujourd'hui. Je voulais vous donner une petite mise à jour sur l'avancement.",
    "es": "Buenos días, espero que estés bien hoy. Quería darte una actualización rápida sobre el proyecto.",
}

# Sanity check that the required Italian reference exists
assert Path(REFERENCE_AUDIO["it"]).exists(), (
    f"Missing Italian reference audio at {REFERENCE_AUDIO['it']} — update the path above."
)
print("Config OK.")

In [ ]:
print("Carlo's Italian reference clip:")
display(Audio(REFERENCE_AUDIO["it"]))

## 3. VoxCPM2

https://huggingface.co/openbmb/VoxCPM2 — 2B params, 30 languages, 48kHz output, tokenizer-free.

In [ ]:
from voxcpm import VoxCPM

voxcpm_model = VoxCPM.from_pretrained("openbmb/VoxCPM2")
print("VoxCPM2 loaded.")

In [ ]:
def generate_voxcpm(text, prompt_wav_path, prompt_text=None, out_path="out.wav"):
    """Clone a voice with VoxCPM2 and synthesize `text`.
    Passing prompt_text activates VoxCPM2's stronger 'Ultimate Cloning' mode.
    """
    start = time.time()
    wav = voxcpm_model.generate(
        text=text,
        prompt_wav_path=prompt_wav_path,
        prompt_text=prompt_text,
        cfg_value=2.0,
        inference_timesteps=10,
    )
    elapsed = time.time() - start
    sf.write(out_path, wav, voxcpm_model.tts_model.sample_rate)
    audio_duration = len(wav) / voxcpm_model.tts_model.sample_rate
    rtf = elapsed / audio_duration if audio_duration > 0 else float("nan")
    return out_path, elapsed, audio_duration, rtf

In [ ]:
# Clone from the Italian reference, generate in all four languages (cross-lingual test)
voxcpm_results = {}

for lang, text in TEST_SENTENCES.items():
    out_path = OUTPUT_DIR / f"voxcpm_{lang}.wav"
    prompt_text = REFERENCE_TRANSCRIPT_IT if lang == "it" else None
    path, elapsed, dur, rtf = generate_voxcpm(
        text=text,
        prompt_wav_path=REFERENCE_AUDIO["it"],
        prompt_text=prompt_text,
        out_path=str(out_path),
    )
    voxcpm_results[lang] = {"path": path, "gen_time": elapsed, "audio_dur": dur, "rtf": rtf}
    print(f"[VoxCPM2] {lang}: gen={elapsed:.2f}s  audio={dur:.2f}s  RTF={rtf:.2f}")

In [ ]:
for lang, r in voxcpm_results.items():
    print(f"--- VoxCPM2 / {lang} ---")
    display(Audio(r["path"]))

## 4. Chatterbox Multilingual

https://huggingface.co/ResembleAI/chatterbox — 0.5B, 23 languages, MIT licensed.

In [ ]:
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

chatterbox_model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
print("Chatterbox Multilingual loaded.")

In [ ]:
def generate_chatterbox(text, language_id, audio_prompt_path, out_path="out.wav"):
    """Clone a voice with Chatterbox Multilingual and synthesize `text`."""
    start = time.time()
    wav = chatterbox_model.generate(
        text=text,
        language_id=language_id,
        audio_prompt_path=audio_prompt_path,
    )
    elapsed = time.time() - start
    sr = chatterbox_model.sr
    sf.write(out_path, wav.squeeze().cpu().numpy() if hasattr(wav, "cpu") else wav, sr)
    audio_duration = (wav.shape[-1] if hasattr(wav, "shape") else len(wav)) / sr
    rtf = elapsed / audio_duration if audio_duration > 0 else float("nan")
    return out_path, elapsed, audio_duration, rtf

In [ ]:
# Same test: clone from the Italian reference, generate in all four languages
chatterbox_results = {}

for lang, text in TEST_SENTENCES.items():
    out_path = OUTPUT_DIR / f"chatterbox_{lang}.wav"
    path, elapsed, dur, rtf = generate_chatterbox(
        text=text,
        language_id=lang,
        audio_prompt_path=REFERENCE_AUDIO["it"],
        out_path=str(out_path),
    )
    chatterbox_results[lang] = {"path": path, "gen_time": elapsed, "audio_dur": dur, "rtf": rtf}
    print(f"[Chatterbox] {lang}: gen={elapsed:.2f}s  audio={dur:.2f}s  RTF={rtf:.2f}")

In [ ]:
for lang, r in chatterbox_results.items():
    print(f"--- Chatterbox / {lang} ---")
    display(Audio(r["path"]))

## 5. Speed comparison (Real-Time Factor)

RTF < 1.0 means faster than real-time (good). Lower is faster.

In [ ]:
print(f"{'Lang':<6}{'VoxCPM2 RTF':<16}{'Chatterbox RTF':<16}")
for lang in TEST_SENTENCES:
    v_rtf = voxcpm_results.get(lang, {}).get("rtf", float("nan"))
    c_rtf = chatterbox_results.get(lang, {}).get("rtf", float("nan"))
    print(f"{lang:<6}{v_rtf:<16.2f}{c_rtf:<16.2f}")

## 6. Side-by-side listening

Compare both models on the same language, directly against each other and against the original reference.

In [ ]:
def compare(lang):
    display(Markdown(f"### {lang.upper()}"))
    print("Original reference (Italian clip):")
    display(Audio(REFERENCE_AUDIO["it"]))
    print("VoxCPM2:")
    display(Audio(voxcpm_results[lang]["path"]))
    print("Chatterbox:")
    display(Audio(chatterbox_results[lang]["path"]))

compare("it")

In [ ]:
compare("en")

In [ ]:
compare("fr")

In [ ]:
compare("es")

## What to listen for

- **Similarity**: does it actually sound like Carlo, or a generic voice?
- **Accent accuracy**: for EN/FR/ES output, does the accent stay natural or does it sound "off"?
- **Prosody**: does it sound flat/robotic, or does pitch/pacing feel natural?
- **Artifacts**: clicks, mumbling, or mispronounced words (more likely on cross-lingual generation).
- **Speed**: RTF from section 5 — lower is better for production use.

If a non-Italian language sounds noticeably worse, that's the signal to ask Carlo to record that language directly rather than relying on cross-lingual generation.